<a href="https://colab.research.google.com/github/lmaas37/FDD-Week-2/blob/main/04a_lime_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍋 LIME: Local Interpretable Model-agnostic Explanations

### A hands-on companion
**· ETH Zürich**

Sources: Ribeiro, Singh & Guestrin (2016), *"Why Should I Trust You?"* · Molnar, *Interpretable Machine Learning*, Ch. 14

---

### 🎯 What you will do

| § | Topic | Question you will be able to answer |
|---|---|---|
| 1 | 🧭 Interpretable vs. post-hoc | When is a black box worth the trouble? |
| 2 | 🔬 The mechanism | What are `x`, `x′`, `z′`, `z`? |
| 3 | 🎛️ The proximity kernel `πₓ` | What does "local" actually mean? |
| 4 | ⚠️ Limitations | When does LIME break? |
| 5 | 🧪 Trusting the model | How do single explanations become global understanding? |

> ⏱️ Runtime: about 20 minutes. Everything is synthetic and CPU only.
> 🔑 Every question hides its answer. **Commit to an answer before you click.**

## 0. 🛠️ Setup

In [1]:
!pip install lime -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 6.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
import numpy as np, matplotlib.pyplot as plt, warnings
warnings.filterwarnings("ignore")
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

SEED = 0
BLUE, PETROL, BRONZE, RED = "#215CAF", "#007894", "#8E6713", "#B7352D"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.grid": True,
                     "grid.alpha": .3, "axes.spines.top": False, "axes.spines.right": False})
print("✅ ready")

✅ ready


---
# 1. 🧭 Interpretable model, or post-hoc explanation?

The default position, argued forcefully by Rudin (2019), is that **an interpretable model should be
the first attempt**, and a black box plus an explainer is a fallback that must justify itself.

The justification is almost always **accuracy**. So the honest way to decide is to *measure the gap*.

### 📊 The scenario

A Swiss bank scores loan applications. The table contains four columns:

| feature | meaning |
|---|---|
| `income_kchf` | annual income, thousands of CHF |
| `debt_ratio` | debt to income |
| `years_employed` | employment tenure |
| `phone_call_made` | did a bank officer phone this applicant? |

The label is `approved`. We generate the data ourselves, so we **know the ground truth rule**:
approval genuinely depends on income, debt ratio and tenure. Remember that.

In [4]:
def make_loan_data(n=4000, flip=0.08, seed=SEED):
    """Synthetic loan book. The causal rule uses income, debt_ratio, years_employed."""
    r = np.random.default_rng(seed)
    income = r.normal(60, 18, n).clip(15, 140)
    debt_ratio = r.beta(2.2, 3.0, n)
    years_employed = r.exponential(5, n).clip(0, 35)

    logit = 0.055*(income - 55) - 5.0*(debt_ratio - 0.45) + 0.14*years_employed - 0.6
    approved = (r.random(n) < 1/(1 + np.exp(-logit))).astype(int)

    # A bank officer phones the applicant once the decision is essentially made.
    # It lands in the same table. It is recorded AFTER the outcome.
    phone_call_made = np.where(r.random(n) < flip, 1 - approved, approved).astype(float)

    X = np.column_stack([income, debt_ratio, years_employed, phone_call_made])
    return X, approved

FEATURES = ["income_kchf", "debt_ratio", "years_employed", "phone_call_made"]
HONEST = [0, 1, 2]          # the three causal features
X, y = make_loan_data()
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=SEED, stratify=y)
print(f"{len(Xtr)} train / {len(Xte)} test · approval rate {y.mean():.1%}")

2800 train / 1200 test · approval rate 58.2%


### 🥊 Round 1: the interpretable model against the black box

We use only the three causal features for now, and ask the question that decides everything:
**how much accuracy does the black box actually buy?**

In [5]:
lr_h = LogisticRegression(max_iter=2000).fit(Xtr[:, HONEST], ytr)
gb_h = GradientBoostingClassifier(random_state=SEED).fit(Xtr[:, HONEST], ytr)
dt_h = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(Xtr[:, HONEST], ytr)

print("Honest features only")
print(f"  📗 Logistic regression (interpretable) : {accuracy_score(yte, lr_h.predict(Xte[:, HONEST])):.4f}")
print(f"  📗 Decision tree, depth 3 (interpretable): {accuracy_score(yte, dt_h.predict(Xte[:, HONEST])):.4f}")
print(f"  📕 Gradient boosting (black box)       : {accuracy_score(yte, gb_h.predict(Xte[:, HONEST])):.4f}")
print("\n📗 Logistic regression coefficients (fully readable, no LIME needed):")
for n, c in zip([FEATURES[i] for i in HONEST], lr_h.coef_[0]):
    print(f"     {n:18s} {c:+.4f}")

Honest features only
  📗 Logistic regression (interpretable) : 0.7467
  📗 Decision tree, depth 3 (interpretable): 0.6717
  📕 Gradient boosting (black box)       : 0.7308

📗 Logistic regression coefficients (fully readable, no LIME needed):
     income_kchf        +0.0564
     debt_ratio         -4.5096
     years_employed     +0.1457


### 🔍 Read that result carefully

The interpretable model is **not worse**. On this data it is very slightly *better*.

This is not a quirk of our simulation. On tabular data with meaningful features it is the common case,
and it is precisely Rudin's point: the accuracy tax on interpretability is often **zero**, and people pay
for a black box anyway.

There is no argument for LIME here. Read the coefficients and go home. 🏠

### 🥊 Round 2: now add the fourth column

In [ ]:
gb_all = GradientBoostingClassifier(random_state=SEED).fit(Xtr, ytr)
lr_all = LogisticRegression(max_iter=2000).fit(Xtr, ytr)
acc_all = accuracy_score(yte, gb_all.predict(Xte))
acc_hon = accuracy_score(yte, gb_h.predict(Xte[:, HONEST]))

print(f"  📕 Gradient boosting, ALL 4 features : {acc_all:.4f}   🎉")
print(f"  📕 Gradient boosting, honest 3 only  : {acc_hon:.4f}")
print(f"\n  Improvement: +{100*(acc_all-acc_hon):.1f} accuracy points.")
print("  A committee would ship this model today. 🚀")

> 🚨 **This is the moment the whole notebook turns on.**
>
> We now have a black box that is **19 points better** than any interpretable model we could fit.
> The accuracy argument has been made, and it looks unanswerable. We are going to deploy `gb_all`.
>
> And because it is a black box, we will need LIME to see inside it.
>
> Keep the number **0.92** in your head. We will return to it in section 5.

#### ❓ Question 1.1

A hospital wants to predict 30 day readmission from 40 structured clinical variables. A logistic regression reaches AUC 0.79; a gradient boosting model reaches AUC 0.81. The output is used by clinicians to allocate follow up care. What is the defensible choice?

- **(a)** Gradient boosting with LIME, because 0.81 > 0.79.
- **(b)** Logistic regression, because two AUC points do not buy back the loss of auditability in a high stakes, individually accountable setting.
- **(c)** Gradient boosting without any explanation, because clinicians trust accuracy.
- **(d)** Either, since LIME reproduces the black box exactly.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

This is exactly the situation Rudin describes. The gap is small, the stakes are high, and the features are already meaningful clinical concepts. LIME would give you an *approximation* of the model, with its own error, in a setting where you may have to defend a decision to a patient or a regulator. **(d) is simply false**: LIME is a local approximation and its fidelity is routinely around R² ≈ 0.3, as you will measure in section 3.

</details>

#### ❓ Question 1.2

For which task is post-hoc explanation the *honest* choice rather than a shortcut?

- **(a)** Predicting churn from 12 CRM columns.
- **(b)** Credit scoring from 20 financial attributes.
- **(c)** Classifying diabetic retinopathy from retinal photographs.
- **(d)** Predicting house prices from 15 property attributes.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (c)**

Images. The paper makes the argument itself: a supersparse linear model with 5 to 10 features is unsuitable for perceptual data, and interpretability there costs flexibility, accuracy or efficiency. Nobody classifies photographs with a decision tree. **(a), (b), (d) are all tabular with meaningful features**, which is precisely where the interpretable model usually keeps up.

</details>

#### ❓ Question 1.3

Which statement about decision trees is correct?

- **(a)** Decision trees are linear models, which is why LIME can use them as surrogates.
- **(b)** Decision trees are always interpretable, regardless of depth.
- **(c)** Decision trees are not linear; they are piecewise constant. They belong to LIME's family G because they are readable, not because they are linear.
- **(d)** Decision trees cannot be black boxes.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (c)**

The paper lists G as *linear models, decision trees, or falling rule lists*, and defines Ω(g) as the **depth** for trees and the **number of non-zero weights** for linear models. Two different complexity measures, because they are two different kinds of object. **(b) and (d) are the same mistake in reverse**: a depth-30 tree is unreadable, and Molnar's own text example uses a *deep decision tree as the black box*. Interpretable does not mean small, and small does not mean linear.

</details>

---
# 2. 🔬 The mechanism: `x`, `x′`, `z′`, `z`

LIME carries **two representations of the same instance**.

| | lives in | who reads it |
|---|---|---|
| `x` | `ℝᵈ` raw values, pixels, embeddings | 🤖 the model. This is the **domain of f** |
| `x′` | `{0,1}ᵈ′` a row of switches | 🧑 the person. This is the **domain of g** |

Four objects, and only four:

| symbol | what it is |
|---|---|
| `x` | the instance, in raw form |
| `x′` | the same instance as switches. **Always all ones**: nothing has been removed yet |
| `z′` | a perturbation: some switches flipped off |
| `z` | `z′` translated back into raw form, so that `f` can eat it |

> 🔑 **Why is `x′` all ones?** Because we defined `0 = removed`. The instance with nothing removed can
> only be a row of ones. It carries no information, and that is its job: it is the **origin** we measure
> distance from. `z′` says what changed; `x′` says what "unchanged" means.

### 👀 Watch the four objects appear

In [ ]:
rng = np.random.default_rng(1)
probs_all = gb_all.predict_proba(Xte)[:, 1]
i0 = int(np.argmin(np.abs(probs_all - 0.05)))
x = Xte[i0]

print("x   (raw)        :", dict(zip(FEATURES, x.round(3))))
print("f(x)             :", round(probs_all[i0], 4), "→ DENIED ❌")
print("x'  (switches)   :", np.ones(4, dtype=int), "  ← all ones: nothing removed\n")

def recover_z(zp, x, Xtrain, rng):
    """Turn a switch pattern into a raw input. 1 = keep his value, 0 = resample from the data."""
    donors = Xtrain[rng.integers(0, len(Xtrain), size=len(x)), np.arange(len(x))]
    return np.where(zp == 1, x, donors)

print(f"{'z prime':>14} | {'z  (actual input to f)':>44} | {'f(z)':>7} | {'D':>4} | {'pi':>5}")
print("-"*84)
for zp in [np.array([1,1,1,1]), np.array([0,1,1,1]), np.array([1,1,1,0]),
           np.array([1,0,1,1]), np.array([0,0,1,0])]:
    z = recover_z(zp, x, Xtr, rng)
    fz = gb_all.predict_proba(z.reshape(1, -1))[0, 1]
    D = np.linalg.norm(zp - np.ones(4)); pi = np.exp(-D**2/(0.75*np.sqrt(4))**2)
    print(f"{str(zp):>14} | {str(z.round(2)):>44} | {fz:7.4f} | {D:4.2f} | {pi:5.2f}")

In [ ]:
# ⚠️ The same z' run five times. Watch the LAST column.
zp = np.array([1, 1, 1, 0])          # only phone_call_made switched OFF
rng2 = np.random.default_rng(11)
print("z' =", zp, " (identical every time)\n")
print(f"{'run':>4} | {'z  (recovered)':>42} | {'f(z)':>7}")
print("-"*60)
for t in range(5):
    z = recover_z(zp, x, Xtr, rng2)
    print(f"{t+1:4d} | {str(z.round(2)):>42} | {gb_all.predict_proba(z.reshape(1,-1))[0,1]:7.4f}")
print("\n🔑 Same switch pattern. Different raw input. Different prediction.")
print("   For TABULAR data, z' does NOT determine z. This noise is irreducible,")
print("   and no surrogate of any kind could remove it.")

### 🧐 Two things to notice in that table

1. 🎯 **Flipping `phone_call_made` off moves the prediction enormously.** One row of this table is already
   an explanation in miniature. LIME simply does it thousands of times so the answer does not depend on
   one lucky draw.

2. ⚠️ **`z′` does not determine `z`.** Switching a feature off resamples it *at random*. The same switch
   pattern run twice gives two different raw inputs, and therefore two different `f(z)`.

Point 2 has a consequence that is easy to miss and important later:

> `g(z′)` **cannot** predict `f(z)` exactly, because `z′` does not contain enough information to do so.
> There is irreducible noise in the target. This is a large part of why LIME's local R² is rarely high.

### 🖼️ 📝 The same recipe on images and text

Only the definition of a switch changes. The recovery step differs in one crucial way:

| | one switch is | `z′ⱼ = 0` means | how `z` is built | deterministic? |
|---|---|---|---|---|
| 📝 **Text** | one word of *this* sentence | the word is deleted | remove it, hand `f` the shorter string | ✅ yes |
| 🖼️ **Image** | one super-pixel (about 50 patches) | the patch is hidden | **overwrite** those pixels with grey or the patch mean | ✅ yes |
| 📊 **Tabular** | one feature | a *different* value | draw a replacement from the training distribution | ❌ **random** |

> 🔑 A tensor cannot have a hole and a person cannot have "no income". For images we **overwrite**;
> for tabular we **substitute**. Only text can truly delete.

In [ ]:
# 🖼️ The image mechanism, made visible. No pretrained model needed.
img = np.ones((64, 64, 3)) * np.array([0.85, 0.91, 0.96])
yy, xx = np.mgrid[0:64, 0:64]
img[(xx-40)**2/90 + (yy-38)**2/40 < 1] = [0.45, 0.45, 0.48]      # the "animal"
img[yy > 52] = [0.97, 0.97, 0.99]                                 # "snow"

segments = (yy//16)*4 + (xx//16)                                   # 16 grid super-pixels
d_img = segments.max() + 1

def hide(img, segments, zp):
    """z'_j = 0  ->  overwrite super-pixel j with its own mean colour."""
    out = img.copy()
    for j in np.where(zp == 0)[0]:
        m = segments == j
        out[m] = img[m].mean(axis=0)
        out[m] = out[m]*0.35 + 0.65   # visibly greyed
    return out

r = np.random.default_rng(3)
fig, axes = plt.subplots(1, 5, figsize=(13, 2.9))
axes[0].imshow(img); axes[0].set_title("x  (original)\nx′ = [1,1,...,1]", fontsize=9)
for k in range(1, 5):
    zp = r.integers(0, 2, d_img)
    axes[k].imshow(hide(img, segments, zp))
    axes[k].set_title(f"z′ = {''.join(map(str, zp[:8]))}...\n{(zp==0).sum()} patches hidden", fontsize=9)
for a in axes:
    a.set_xticks([]); a.set_yticks([]); a.grid(False)
plt.suptitle("🖼️  Image: the switch vector is a super-pixel mask", y=1.06, fontsize=11)
plt.tight_layout(); plt.show()

In [ ]:
# 📝 The text mechanism. The paper: draw HOW MANY words to remove uniformly, then WHICH ones uniformly.
sentence = "the plot was predictable and dull".split()
d_txt = len(sentence)
r = np.random.default_rng(1)

print(f"x   : \"{' '.join(sentence)}\"")
print(f"x'  : {np.ones(d_txt, dtype=int)}   (d′ = {d_txt}: one switch per word)\n")
for _ in range(4):
    k = r.integers(1, d_txt + 1)                       # roll 1: how many
    off = r.choice(d_txt, size=k, replace=False)       # roll 2: which
    zp = np.ones(d_txt, dtype=int); zp[off] = 0
    z = " ".join(w for w, s in zip(sentence, zp) if s == 1)
    print(f"z'  : {zp}  ->  z : \"{z}\"")
print("\n🔑 LIME hands f the shorter STRING. f computes its own embedding internally.")
print("   LIME never touches embeddings. That is what model-agnostic means.")

#### ❓ Question 2.1

For a sentence of 6 words classified by a model that uses 768 dimensional embeddings, what are d and d′?

- **(a)** d = 6 and d′ = 768.
- **(b)** d = 768 and d′ = 6.
- **(c)** d = d′ = 768.
- **(d)** d = d′ = 6.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

`d` is the model's representation (768 embedding dimensions); `d′` is the interpretable one (6 word switches). They differ because they were chosen by different parties for different readers. Note that **d′ changes per instance**: a 20 word sentence has d′ = 20. This is only permissible because the explanation is *local*, valid for this instance alone. You could never do this globally.

</details>

#### ❓ Question 2.2

Which statement about the interpretable representation of TABULAR data is correct?

- **(a)** x′ cannot be binary for tabular data, so LIME uses the raw values.
- **(b)** d′ < d, because tabular features get compressed into groups.
- **(c)** d′ = d, and x′ is still binary: the switch encodes whether the feature sits in the same bin as the instance.
- **(d)** Tabular data does not need an interpretable representation at all.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (c)**

The formalism survives. The default implementation discretises each feature into quartiles, and the switch is 1 when a sampled value lands in the **same bin** as the instance. So d′ = d and x′ is all ones, exactly as for text and images. **(d) is tempting and half right**: `income = 46 kCHF` really is already a human concept, unlike a pixel. But you still need the binary encoding to define *perturbation* and to make each coefficient mean 'the effect of keeping this value'.

</details>

#### ❓ Question 2.3

Why is x′ always a vector of ones?

- **(a)** It is an arbitrary convention chosen by the authors.
- **(b)** Because 0 was defined to mean 'removed', so the instance with nothing removed can only be all ones.
- **(c)** Because all features are equally important before we start.
- **(d)** Because the binary vector stores the instance's normalised values.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

Once `0 = removed` is fixed, all ones is **forced**. There is no other possibility. It is not a convention to accept on faith, it is a consequence. And yes, x′ carries no information about the instance: that is its purpose. It is the origin of the neighbourhood. **(d) confuses x′ with x**: the instance's actual values live downstairs, in raw space.

</details>

---
## ✍️ Exercise 2.1: implement LIME from scratch

Everything from the lecture, in about fifteen lines. Fill in the five gaps.

**The recipe**
1. 🎲 **Perturb**: draw `N` switch patterns `z′`, each entry a coin flip. Force row 0 to be `x′` (all ones).
2. 🔄 **Recover**: build `z` from each `z′`. Keep the value where the switch is on, resample where it is off.
3. 📞 **Query**: ask the black box for `f(z)`.
4. ⚖️ **Weight**: `D` = distance from `x′` **in switch space**; `π = exp(−D²/σ²)`.
5. 📏 **Fit**: weighted ridge of `f(z)` on `z′`. **Return the coefficients.**

**🧰 Toolbox** — everything you need is already in scope:

| you have | what it gives you |
|---|---|
| `rng.integers(low, high, size=…)` | random ints — the 0/1 coin flips for `z′` |
| `donors` | replacement values pre-drawn from the training data, one per cell (for step 2) |
| `predict_proba(Z)` | array `(N, 2)`; column `1` is `f(z)` |
| `np.where(mask, a, b)` | pick `a` where `mask` is true, else `b` |
| `np.linalg.norm(…, axis=1)`, `np.ones(d)`, `np.exp(…)` | the distance `D` and the kernel `π` |
| `Ridge(alpha=alpha).fit(X, y, sample_weight=…)` | weighted least squares; read `.coef_` |


In [ ]:
def lime_from_scratch(x, predict_proba, Xtrain, sigma=None, N=5000, seed=0, alpha=1.0):
    """Returns (weights, intercept, R2, g(x'), SS_tot)."""
    rng = np.random.default_rng(seed)
    d = len(x)
    sigma = sigma if sigma is not None else 0.75*np.sqrt(d)

    # 1️⃣ PERTURB  — draw N switch patterns z', each entry a coin flip (0/1),
    #               then force row 0 to be x' (all ones).
    Zp = ...                                      # shape (N, d), entries in {0, 1}
    Zp[0] = ...                                   # x' is always all ones

    # 2️⃣ RECOVER  — build z from each z': keep x where the switch is on (1),
    #               resample a donor value from the training data where it is off (0).
    donors = Xtrain[rng.integers(0, len(Xtrain), size=(N, d)), np.arange(d)]
    Z = ...                                       # np.where(...) over Zp, x, donors
    Z[0] = x

    # 3️⃣ QUERY   — ask the black box for f(z): the P(class = 1) column.
    fz = ...

    # 4️⃣ WEIGHT  — distance D from x' in SWITCH space (not raw space),
    #               then the proximity kernel pi = exp(-D^2 / sigma^2).
    D = ...                                       # ||Zp - ones(d)|| per row
    pi = ...

    # 5️⃣ FIT     — weighted ridge of f(z) on z'; the coefficients are the explanation.
    g = ...                                       # weighted ridge of fz on Zp; see Toolbox

    # diagnostics
    pred = g.predict(Zp)
    ybar = np.average(fz, weights=pi)
    ss_tot = (pi*(fz - ybar)**2).sum()
    ss_res = (pi*(fz - pred)**2).sum()
    r2 = 1 - ss_res/ss_tot if ss_tot > 1e-10 else np.nan
    return g.coef_, g.intercept_, r2, g.predict(np.ones((1, d)))[0], ss_tot


w, b, r2, gx, sst = lime_from_scratch(x, gb_all.predict_proba, Xtr)
print("🍋 LIME from scratch, applicant", i0, "\n")
for n, v in zip(FEATURES, w):
    bar = "█" * int(abs(v)*40)
    print(f"  {n:18s} {v:+.4f}  {bar}")
print(f"\n  R² = {r2:.3f}   g(x') = {gx:.4f}   vs   f(x) = {probs_all[i0]:.4f}")

> 🔬 **Sanity check against the reference implementation.** Ours uses the simplified
> "1 = keep the exact value" rule from the lecture; the `lime` package bins features into quartiles first,
> so the numbers differ slightly. **The ranking is what should agree.**

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
explainer = LimeTabularExplainer(Xtr, feature_names=FEATURES, class_names=["denied", "approved"],
                                 categorical_features=[3], discretize_continuous=True, random_state=SEED)
exp = explainer.explain_instance(x, gb_all.predict_proba, num_features=4, num_samples=5000)
print("📦 lime package:")
for f_, v in exp.as_list(label=1):
    print(f"   {f_:36s} {v:+.4f}")
print(f"\n   R² (exp.score) = {exp.score:.3f}   g(x') (exp.local_pred) = {exp.local_pred[0]:.4f}")
print("\n🔑 Both agree: phone_call_made dominates. Hold that thought until section 5.")

---
# 3. 🎛️ The proximity kernel `πₓ`: what does "local" mean?

$$\pi_x(z) = \exp\!\left(-\frac{D(x,z)^2}{\sigma^2}\right)$$

`σ` is the **kernel width**. It is the single most consequential number in LIME, and
**nobody knows how to choose it.** The default is a rule of thumb: `σ = 0.75 · √d`.

Sweep it and watch what happens. 👇

In [ ]:
sigmas = [0.05, 0.1, 0.3, 0.75, 1.5, 3.0, 10.0]
rows = []
for s in sigmas:
    ww, _, rr, _, sst_ = lime_from_scratch(x, gb_all.predict_proba, Xtr, sigma=s)
    rows.append((s, ww, rr, sst_))

print(f"{'sigma':>6} | " + " | ".join(f"{n[:11]:>11}" for n in FEATURES) + f" | {'R²':>6} | {'SS_tot':>8}")
print("-"*88)
for s, ww, rr, sst_ in rows:
    flag = "  ⬅️ DEFAULT" if abs(s - 0.75*np.sqrt(4)) < 1e-9 else ""
    print(f"{s:6.2f} | " + " | ".join(f"{v:+11.4f}" for v in ww) +
          f" | {rr:6.3f} | {sst_:8.4f}{flag}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))
arr = np.array([r[1] for r in rows])
for j, n in enumerate(FEATURES):
    axes[0].plot(sigmas, arr[:, j], "o-", label=n, lw=1.8)
axes[0].set_xscale("log"); axes[0].axvline(1.5, color="grey", ls="--", lw=1)
axes[0].set_xlabel("kernel width σ  (log)"); axes[0].set_ylabel("LIME weight")
axes[0].set_title("🎛️ The explanation is a function of σ"); axes[0].legend(fontsize=7)
axes[0].axhline(0, color="black", lw=.8)

axes[1].plot(sigmas, [r[3] for r in rows], "o-", color=RED, lw=1.8)
axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("kernel width σ  (log)"); axes[1].set_ylabel("weighted SS_tot  (log)")
axes[1].set_title("⚠️ Below σ ≈ 0.3 there is no variance left to explain")
axes[1].axvline(1.5, color="grey", ls="--", lw=1)
plt.tight_layout(); plt.show()

### 🚨 The trap in that table

Look at the row `σ = 0.05`:

```
weights = 0.0000  0.0000  0.0000  0.0000        R² = nan (or 1.0)      SS_tot ≈ 0
```

**Every weight is exactly zero.** Why? With `σ` that small, the only samples with non-zero weight are
those where `z′ = x′` exactly. Those all recover `z = x`, so `f(z) = f(x)` for every one of them.
**The target is constant.** A regression on a constant target returns a flat line: intercept only,
all slopes zero.

And now the dangerous part. Measure fidelity by **agreement at the point** and this surrogate is *perfect*:
`g(x′) = f(x)` exactly. 🎉

> ### ⚠️ Perfect fidelity. Zero information.
>
> `R² ≈ 0` (or `nan`, or `1.0`, depending on the numerics of `0/0`) is **ambiguous**. It means either
> *"g failed to explain the variance"* or *"there was no variance to explain."*
> Two opposite situations, one number.
>
> **Always check `SS_tot` before interpreting `R²`.**

At the other end, `σ = 10`, every sample counts equally, the neighbourhood is the whole dataset, and you
are no longer explaining a prediction. You are fitting a global linear model and calling it local.

#### ❓ Question 3.1

You run LIME and every reported weight is 0.000. What is the most likely explanation?

- **(a)** The model is broken.
- **(b)** The features are all irrelevant to the model globally.
- **(c)** In the neighbourhood defined by πₓ, f is constant: no perturbation changed the prediction, so there is nothing to fit.
- **(d)** The Lasso removed all features because K was set to 0.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (c)**

This is real and it is in Molnar's book. For the non spam comment *'PSY is a good guy'*, **every weight is 0.000**, because no matter which word is removed, the predicted class stays the same. Crucially the explanation is **not wrong**: locally, f really is constant, and LIME correctly reports no local sensitivity. It is simply useless to whoever asked 'why?'. **(b) is a different claim** and does not follow: a feature can matter enormously elsewhere and not here.

</details>

#### ❓ Question 3.2

Widening σ from 0.75 to 10 changes the explanation. What has actually happened?

- **(a)** The explanation became more faithful, because more data was used.
- **(b)** The neighbourhood grew until it covered the dataset, so we now approximate the GLOBAL model, and local fidelity drops.
- **(c)** Nothing meaningful; σ only affects runtime.
- **(d)** The black box changed.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

Watch the R² column fall as σ grows. The line is being asked to fit an ever larger region with the same single slope, so it fits the neighbourhood of x ever worse. σ is the dial between *local and uninformative* and *global and unfaithful*, and **there is no principled way to set it**. Molnar names kernel width as the biggest unsolved problem of LIME.

</details>

---
# 4. ⚠️ Limitations

## 4.1 🎲 Is LIME stable? The sources disagree.

> 📄 **The paper (§3.3):** LIME is *fairly robust to sampling noise, since the samples are weighted by πₓ.*
>
> 📘 **Molnar (Limitations):** *instability is a really big problem*; repeated sampling produces different
> explanations, and he recommends using LIME *only with great care*.

They cannot both be unconditionally true. **Settle it with an experiment**: re-run LIME on the *same*
instance with different random seeds, and vary `N`.

In [ ]:
Ns = [30, 60, 120, 500, 2000, 5000]
print(f"{'N':>6} | {'sd(phone)':>10} | {'sd(income)':>11} | {'top-1 agreement':>16}")
print("-"*54)
stab = []
for N_ in Ns:
    W = np.array([lime_from_scratch(x, gb_all.predict_proba, Xtr, N=N_, seed=s)[0]
                  for s in range(12)])
    top = [FEATURES[k] for k in np.argmax(np.abs(W), axis=1)]
    agree = max(top.count(t) for t in set(top))/len(top)
    stab.append((N_, W[:, 3].std(), W[:, 0].std(), agree))
    print(f"{N_:6d} | {W[:,3].std():10.4f} | {W[:,0].std():11.4f} | {agree:15.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot([s[0] for s in stab], [s[1] for s in stab], "o-", color=RED, lw=2, label="sd of phone_call_made weight")
ax.plot([s[0] for s in stab], [s[2] for s in stab], "o-", color=BLUE, lw=2, label="sd of income weight")
ax.set_xscale("log"); ax.set_xlabel("N  (number of perturbations, log)")
ax.set_ylabel("std. dev. across 12 seeds")
ax.set_title("🎲 Instability is a function of N, not a property of LIME")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 🧠 The honest conclusion

**Both sources are right, in different regimes.**

At `N = 30` the explanation genuinely wobbles and the top feature changes between seeds. By `N ≈ 120`
the ranking is stable, and by `N = 5000` the coefficients agree to two decimals. The paper's claim holds
**when N is large relative to d′**. Molnar's warning bites when it is not, and in realistic settings
`d′` is not 4 but 50 super-pixels or 30 words, where `2^d′` corners are being probed by a few thousand
samples.

> 🔑 **What to take away:** "Is LIME stable?" is not a yes or no question. It is a question about
> **how many samples you drew, in how many dimensions, and how wide your kernel was.**
> Report those numbers, or your explanation is not reproducible.

## 4.2 🌐 Sampling off the data manifold

`LimeTabularExplainer` draws each feature **independently** from its own training distribution, which
destroys the correlations between them. It also samples around the **training data's mass centre**, not
around your instance.

In [ ]:
r = np.random.default_rng(SEED)
n_show = 4000
fake = np.column_stack([r.normal(Xtr[:,0].mean(), Xtr[:,0].std(), n_show),
                        r.normal(Xtr[:,2].mean(), Xtr[:,2].std(), n_show)])

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharex=True, sharey=True)
axes[0].scatter(Xtr[:,0], Xtr[:,2], s=4, alpha=.25, color=BLUE)
axes[0].set_title("✅ Real applicants"); axes[0].set_xlabel("income_kchf"); axes[0].set_ylabel("years_employed")
axes[1].scatter(fake[:,0], fake[:,1], s=4, alpha=.25, color=RED)
axes[1].axhline(0, color="black", lw=1)
axes[1].set_title("❌ LIME's perturbations (independent marginals)"); axes[1].set_xlabel("income_kchf")
n_imp = (fake[:,1] < 0).sum()
plt.tight_layout(); plt.show()
print(f"⚠️  {n_imp} of {n_show} perturbed applicants have NEGATIVE years of employment ({n_imp/n_show:.1%}).")
print("    The black box was never trained on such people. It will still answer.")
print("    LIME then fits a line through those answers and calls it an explanation.")

> 🚨 **This is the attack surface.** Slack et al. (2020) showed LIME explanations can be **deliberately
> manipulated to hide a model's biases**. The attack works precisely because perturbed points are off
> distribution and therefore *detectable*: build a model that behaves badly on real data and innocently
> on anything that smells synthetic, and LIME only ever sees the synthetic ones.
>
> 🤔 **Molnar's counter-argument, which is fair:** sampling broadly *increases the probability that some
> sample predictions differ from the instance*, so that LIME can learn **at least some** explanation.
> Sample too tightly and you land in section 3's trap: all weights zero. It is a trade, not a blunder.

#### ❓ Question 4.1

Two applicants differ only in the third decimal of debt_ratio, and receive noticeably different LIME explanations. What has gone wrong?

- **(a)** The black box is discontinuous at that point.
- **(b)** Nothing is 'wrong' in the code: the explanations come from two different random samples, and the sampling variance exceeds the real difference.
- **(c)** LIME has a bug.
- **(d)** The kernel width was too large.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

This is the Alvarez-Melis and Jaakkola (2018) result that Molnar cites: explanations of two very close points varied greatly. The instance barely moved; the **random draw** moved. It is a variance problem, and the fix is more samples (or reporting the variance). **(a) is possible in principle** but far less likely than the mundane explanation, and you can tell them apart by re-running with more samples: if the difference survives, it is the model; if it vanishes, it was noise.

</details>

#### ❓ Question 4.2

Your LIME explanation reports weighted R² = 0.31. What is the correct reading?

- **(a)** The explanation is wrong and should be discarded.
- **(b)** The surrogate captures about 31% of how f varies locally: the ranking is usable, but most local behaviour is unexplained. This is normal for LIME.
- **(c)** The model is only 31% accurate.
- **(d)** 31% of the features are important.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

R² = 1 − SS_res/SS_tot, the fraction of *local variance in f's output* that the line reproduces. Around 0.3 is typical and honest. Two reasons it is not higher: a straight line cannot copy a curved surface, **and** for tabular data z′ does not determine z, so part of the variance is irreducible noise no model could capture. **(c) confuses fidelity with accuracy**: R² says nothing about whether f is right, only about whether g resembles f.

</details>

---
# 5. 🧪 Trusting a prediction vs. trusting the model

The paper separates two questions that get confused constantly:

| | question |
|---|---|
| 🔎 **Trusting a prediction** | *Do I act on this one output?* |
| 🏛️ **Trusting a model** | *Do I deploy this thing?* |

We have been answering the first. A participant will reasonably ask: **"why do we care about one
prediction when we have thousands of records?"**

The paper's answer is **SP-LIME** (submodular pick): explain a handful of instances chosen so that
together they **cover as much of the model's behaviour as possible**, while avoiding instances with
similar explanations. Single explanations are the atoms; the pick step is the molecule. 🧬

### 🕵️ First: is our 92% model trustworthy?

In [ ]:
sel = np.random.default_rng(SEED).choice(len(Xte), 60, replace=False)
W = np.array([lime_from_scratch(Xte[k], gb_all.predict_proba, Xtr, N=2000, seed=SEED)[0] for k in sel])

print("🔎 LIME weights across 60 individual predictions\n")
print(f"{'feature':20s} {'mean |w|':>10} {'top-1 driver':>14}")
print("-"*48)
imp = np.abs(W).mean(axis=0)
top1 = np.argmax(np.abs(W), axis=1)
for j, n in enumerate(FEATURES):
    print(f"{n:20s} {imp[j]:10.4f} {(top1==j).mean():13.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 3.4))
order = np.argsort(imp)
cols = [RED if FEATURES[j] == "phone_call_made" else BLUE for j in order]
ax.barh([FEATURES[j] for j in order], imp[order], color=cols)
ax.set_xlabel("mean |LIME weight| over 60 explained predictions")
ax.set_title("🚨 SP-LIME style global view: one feature runs the model")
plt.tight_layout(); plt.show()

### 💣 The reveal

Go back to section 1 and read the two numbers again:

| model | test accuracy |
|---|---|
| 📕 Gradient boosting, **all 4 features** | **0.92** |
| 📗 Logistic regression, honest features | 0.75 |

The black box looked **19 points better**, and that was the entire argument for using it.

It was not a better model. It had found **`phone_call_made`**, a column recorded *after* the decision was
made. On the training table it predicts the outcome almost perfectly. In production, at the moment a new
application arrives, **nobody has phoned anyone yet**. The column is empty. The model is worth 0.

> 🎯 **No aggregate metric could have told you this.** Accuracy said 0.92. Cross validation said 0.92.
> The held out test set said 0.92. Every dataset level number **agreed and every one of them was useless**,
> because the model was *getting the answers right*.
>
> Only looking at individual predictions revealed *why*.

This is exactly the paper's headline experiment: an SVM on 20 newsgroups reaches **94% held out accuracy**,
and the explanations show it keys on the words *"Posting"*, *"Host"* and *"Re"*, which have nothing to do
with the topic. Right answers, worthless reasoning. And the husky/wolf classifier, which predicts **"wolf"
whenever there is snow**, regardless of the animal.

### 🧭 The 2x2 you should carry out of this notebook

Correctness and explanation quality are **independent axes**. All four cells are real.

| f correct? | explanation sensible? | diagnosis | example |
|:---:|:---:|---|---|
| ✅ | ✅ | trust it | |
| ✅ | ❌ | **right for the wrong reasons.** Will not generalise. **The dangerous cell.** | 20 newsgroups · **our loan model** |
| ❌ | ✅ | looked at the right thing, still lost. A genuinely hard case | Molnar's bread predicted as "bagel" |
| ❌ | ❌ | broken | husky/wolf → snow |

> 📏 **Accuracy only reads the rows. LIME reads the columns.**
> That is the argument of the entire paper in one line, and it is why the title is a question about
> *trust* rather than about *accuracy*.

#### ❓ Question 5.1

Our gradient boosting model scores 0.92 on a held out test set and its LIME explanations are dominated by a leaked column. Which cell of the 2x2 is it in, and why is that the dangerous one?

- **(a)** Correct + sensible. It is accurate, so it is fine.
- **(b)** Correct + nonsense. It is dangerous because every aggregate metric reports success, so nothing in the standard workflow can catch it.
- **(c)** Wrong + sensible. The model is simply mistaken.
- **(d)** Wrong + nonsense. It is broken and accuracy would reveal that.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

The model is **getting the answers right**, which is exactly why accuracy, cross validation and the test set all fail to raise an alarm. They are structurally incapable of it. The failure is not in the predictions, it is in the *reasoning*, and reasoning is the one thing an aggregate metric never sees. This is the single strongest argument for explaining individual predictions.

</details>

#### ❓ Question 5.2

A participant asks: 'why explain one prediction when we have 50,000 records?' What is the best answer?

- **(a)** We do not; LIME explains the whole model at once.
- **(b)** Because individual predictions are the only thing that can be explained faithfully (locality is what makes the surrogate valid), and SP-LIME then picks a representative handful to build global understanding.
- **(c)** Because 50,000 records are too many to compute on.
- **(d)** Because global explanations do not exist.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

**Concede the point, then reveal that the authors conceded it too.** The paper explicitly separates trusting a prediction from trusting a model, and builds the pick step to address the second via submodular optimisation, selecting instances with **non redundant** explanations because a human can only look at a few. So: single predictions are the atoms, not the goal. **(d) is false** (permutation importance, PDPs and surrogate trees are all global methods) and **(c) is a computational excuse for a conceptual point**.

</details>

#### ❓ Question 5.3

LIME is run on a prediction that the model got WRONG. What happens?

- **(a)** LIME fails, because it needs the correct label to compute the explanation.
- **(b)** Nothing changes mechanically. LIME never sees the true label; it explains why f said what it said, which for a mistake is exactly what you want.
- **(c)** LIME returns the features that would have produced the right answer.
- **(d)** The R² becomes negative.

<details>
<summary>💡 <b>Reveal answer</b></summary>

**Correct: (b)**

Look at Algorithm 1's requirements: classifier f, N, instance x, x′, kernel πₓ, K. **There is no y.** LIME is structurally incapable of knowing whether f was right. It is a mirror, not a judge, and that is why it works for debugging: the husky/wolf explanation is an explanation of a *mistake*, and it is the most valuable output in the paper. **(c) describes counterfactual explanations**, a different method (Molnar Ch. 15), which answer 'what would have to change?' rather than 'what drove this?'

</details>

---
# 6. 📋 The final explanation

Everything the method produces, for one applicant. Note what the deliverable actually is:
**`w`, one number per switch.** Algorithm 1's last line is `return w`. Not `return g`.

> 🔑 We do not use the simple model to predict anything. We fit it, we throw it away,
> and we keep the coefficients.

In [ ]:
i_final = i0
x_final = Xte[i_final]
w, b, r2, gx, sst = lime_from_scratch(x_final, gb_all.predict_proba, Xtr, N=5000, seed=SEED)

print("="*66)
print(f"  APPLICANT #{i_final}")
print("="*66)
for n, v in zip(FEATURES, x_final):
    print(f"    {n:20s} = {v:8.3f}")
print(f"\n    f(x) = P(approved) = {gb_all.predict_proba(x_final.reshape(1,-1))[0,1]:.4f}"
      f"  →  {'APPROVED ✅' if gb_all.predict(x_final.reshape(1,-1))[0]==1 else 'DENIED ❌'}")

print("\n" + "="*66)
print("  🍋 LIME EXPLANATION   w = one number per switch")
print("     each weight = the effect of KEEPING this feature at his value")
print("="*66)
for n, v in sorted(zip(FEATURES, w), key=lambda t: -abs(t[1])):
    arrow = "▶ toward APPROVED" if v > 0 else "◀ toward DENIED  "
    print(f"    {n:20s} {v:+.4f}   {arrow}  {'█'*int(abs(v)*45)}")

print("\n" + "-"*66)
print("  🔬 FIDELITY CHECK   (not the explanation: the licence to believe it)")
print("-"*66)
print(f"    g(x')            = {gx:+.4f}")
print(f"    f(x)             = {gb_all.predict_proba(x_final.reshape(1,-1))[0,1]:+.4f}")
print(f"    weighted R²      = {r2:.3f}")
print(f"    weighted SS_tot  = {sst:.4f}   {'⚠️ near zero: R² is meaningless' if sst < 1e-3 else '✅ there is variance to explain'}")
print(f"    N                = 5000      σ = {0.75*np.sqrt(4):.2f}")
print("-"*66)
print("\n  ⚠️  This explains ONE prediction of THIS model. Not the model. Not the world.")

---
## 🎓 Summary

| # | Lesson |
|---|---|
| 1️⃣ | On tabular data with meaningful features, the **interpretable model usually keeps up**. Measure the gap before reaching for a black box. |
| 2️⃣ | LIME carries **two representations**. `x` is the domain of `f`; `x′` is the domain of `g`. `x′` is all ones because `0` means removed. |
| 3️⃣ | **`σ` decides what "local" means, and nobody knows how to set it.** Too narrow: every weight is zero. Too wide: you have fitted a global model. |
| 4️⃣ | Stability is **a function of N and d′**, not a property of the method. The paper and Molnar are both right, in different regimes. |
| 5️⃣ | **Accuracy reads the rows; LIME reads the columns.** Our 0.92 model was the dangerous cell of the 2x2, and no aggregate metric could have caught it. |
| 6️⃣ | The deliverable is **`w`**. Fit `g`, throw it away, keep the coefficients. Then check `SS_tot` before you believe `R²`. |

### 📚 Sources
- Ribeiro, M.T., Singh, S., Guestrin, C. (2016). *"Why Should I Trust You?" Explaining the Predictions of Any Classifier.* KDD. [arXiv:1602.04938](https://arxiv.org/pdf/1602.04938)
- Molnar, C. *Interpretable Machine Learning*, [Ch. 14: Local Surrogate (LIME)](https://christophm.github.io/interpretable-ml-book/lime.html)
- Rudin, C. (2019). *Stop explaining black box models for high stakes decisions.* Nature Machine Intelligence.
- Alvarez-Melis, D., Jaakkola, T. (2018). *On the robustness of interpretability methods.*
- Slack, D. et al. (2020). *Fooling LIME and SHAP: adversarial attacks on post hoc explanation methods.*